In [1]:
pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 11.1 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import duckdb

In [3]:
import pandas as pd

In [4]:
import time

In [5]:
# Criar uma conexão DuckDB persistente (o arquivo .duckdb armazena
# metadados e tabelas locais; os dados Parquet permanecem no MinIO)
con = duckdb.connect("../data/curso_analytics.duckdb")
# Instalar e carregar a extensão httpfs (HTTP File System )
# Necessária para acessar arquivos em S3/MinIO
con.execute("INSTALL httpfs;" )
con.execute("LOAD httpfs;" )
# Configurar as credenciais do MinIO
# O DuckDB usa as mesmas configurações do protocolo S3

### Se estiver executando a partir da instalação python da máquina host:
#con.execute("""
#    SET s3_endpoint = 'localhost:9000';
#    SET s3_access_key_id = 'minioadmin';
#    SET s3_secret_access_key = 'minioadmin123';
#    SET s3_use_ssl = false;
#    SET s3_url_style = 'path';
#""")

### Agora, se estiver rodando de um container docker que está na mesma instalação do docker:
con.execute("""
    SET s3_endpoint = 'host.docker.internal:9000';
    SET s3_access_key_id = 'minioadmin';
    SET s3_secret_access_key = 'minioadmin123';
    SET s3_use_ssl = false;
    SET s3_url_style = 'path';
""")
print(" ✅ DuckDB configurado com suporte ao MinIO!")
print(f"   Versão: {duckdb.__version__}")
print(f"   Banco de dados local: ../data/curso_analytics.duckdb")

 ✅ DuckDB configurado com suporte ao MinIO!
   Versão: 1.5.3
   Banco de dados local: ../data/curso_analytics.duckdb


In [6]:
inicio = time.time()
resultado = con.execute("""
    SELECT
        categoria,
        COUNT(*)                          AS total_pedidos,
        ROUND(SUM(valor_total), 2)        AS receita_total,
        ROUND(AVG(valor_total), 2)        AS ticket_medio,
        ROUND(MIN(valor_total), 2)        AS menor_pedido,
        ROUND(MAX(valor_total), 2)        AS maior_pedido
    FROM read_parquet('s3://bronze/ecommerce_sintetico/**/*.parquet')
    WHERE status_pedido = 'concluido'
    GROUP BY categoria
    ORDER BY receita_total DESC
""").df()  # .df() converte o resultado para um DataFrame Pandas
 
duracao = time.time() - inicio
 
print(f"Tempo de execução: {duracao:.3f}s  (lendo 1.000.000 registros do MinIO!)\n")

Tempo de execução: 0.448s  (lendo 1.000.000 registros do MinIO!)



In [7]:
# Célula 3 — Consulta com múltiplas fontes Parquet (JOIN entre datasets)
# DuckDB pode fazer JOIN entre arquivos Parquet de diferentes origens
resultado_join = con.execute("""
    WITH pedidos AS (
        SELECT
            regiao,
            categoria,
            status_pedido,
            valor_total,
            avaliacao_cliente
        FROM read_parquet('s3://bronze/ecommerce_sintetico/**/*.parquet')
    ),
    resumo_regiao AS (
        SELECT
            regiao,
            COUNT(*)                           AS total_pedidos,
            ROUND(SUM(valor_total), 2)         AS receita_total,
            ROUND(AVG(valor_total), 2)         AS ticket_medio,
            ROUND(
                100.0 * COUNT(*) FILTER (WHERE status_pedido = 'cancelado')
       / COUNT(*), 2
            )                                  AS taxa_cancelamento_pct,
            ROUND(
                AVG(avaliacao_cliente)
                FILTER (WHERE avaliacao_cliente IS NOT NULL), 2
            )                                  AS nota_media_clientes
        FROM pedidos
        GROUP BY regiao
    )
    SELECT *
    FROM resumo_regiao
    ORDER BY receita_total DESC
""").df()                

In [8]:
print("Resumo por Região — Receita, Cancelamentos e Satisfação:")
print(resultado_join.to_string(index=False))

Resumo por Região — Receita, Cancelamentos e Satisfação:
      regiao  total_pedidos  receita_total  ticket_medio  taxa_cancelamento_pct  nota_media_clientes
Centro-Oeste         200648   1.107246e+09       5518.35                   9.96                 3.91
         Sul         200055   1.105015e+09       5523.56                   9.87                 3.90
     Sudeste         199800   1.103193e+09       5521.49                   9.97                 3.91
       Norte         199943   1.099606e+09       5499.59                   9.98                 3.91
    Nordeste         199554   1.098873e+09       5506.64                   9.93                 3.90


In [9]:
# Célula 4 — Análise de tendência temporal com window functions SQL
resultado_temporal = con.execute("""
    WITH vendas_mensais AS (
        SELECT
            DATE_TRUNC('month', data_pedido)   AS mes,
            categoria,
            COUNT(*)                           AS pedidos_mes,
            ROUND(SUM(valor_total), 2)         AS receita_mes
        FROM read_parquet('s3://bronze/ecommerce_sintetico/**/*.parquet')
        WHERE status_pedido = 'concluido'
        GROUP BY 1, 2
    ),
    com_crescimento AS (
        SELECT
            mes,
            categoria,
            pedidos_mes,
            receita_mes,
            LAG(receita_mes) OVER (
                PARTITION BY categoria
                ORDER BY mes
            )                                  AS receita_mes_anterior,
            ROUND(
                100.0 * (receita_mes - LAG(receita_mes) OVER (
                    PARTITION BY categoria ORDER BY mes
                )) / NULLIF(LAG(receita_mes) OVER (
                    PARTITION BY categoria ORDER BY mes
                ), 0), 2
            )                                  AS crescimento_pct
        FROM vendas_mensais
    )
    SELECT *
    FROM com_crescimento
    WHERE mes >= '2022-01-01'
      AND mes < '2022-04-01'   -- Primeiros 3 meses para visualização
    ORDER BY categoria, mes
""").df()
print("Crescimento Mensal de Receita por Categoria (Jan-Mar 2022):")
print(resultado_temporal.to_string(index=False))

Crescimento Mensal de Receita por Categoria (Jan-Mar 2022):
       mes   categoria  pedidos_mes  receita_mes  receita_mes_anterior  crescimento_pct
2022-01-01   Alimentos        13367  73087878.22                   NaN              NaN
2022-02-01   Alimentos        12107  66861956.30           73087878.22            -8.52
2022-03-01   Alimentos        13355  74943624.38           66861956.30            12.09
2022-01-01 Eletrônicos        13478  73343279.53                   NaN              NaN
2022-02-01 Eletrônicos        12221  67857475.37           73343279.53            -7.48
2022-03-01 Eletrônicos        13554  74254134.56           67857475.37             9.43
2022-01-01    Esportes        13276  73153747.01                   NaN              NaN
2022-02-01    Esportes        12047  65934309.94           73153747.01            -9.87
2022-03-01    Esportes        13357  73474186.63           65934309.94            11.44
2022-01-01      Livros        13234  73039897.02            

In [10]:
# Célula 5 — Criar views persistentes no DuckDB
# As views ficam salvas no arquivo .duckdb e podem ser reutilizadas
# em sessões futuras sem reconfigurar as credenciais S3
con.execute("""
    CREATE OR REPLACE VIEW vw_pedidos_bronze AS
    SELECT *
    FROM read_parquet('s3://bronze/ecommerce_sintetico/**/*.parquet')
""")

# Verificar as views criadas
views = con.execute("""
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()
print("Views e tabelas disponíveis no DuckDB:")
print(views.to_string(index=False))

Views e tabelas disponíveis no DuckDB:
       table_name table_type
vw_pedidos_bronze       VIEW


In [11]:
# Célula 6 — Consulta usando as views criadas
# Agora podemos consultar os dados como se fossem tabelas locais
resultado_view = con.execute("""
    WITH vendas_mensais AS (
        SELECT
            DATE_TRUNC('month', data_pedido)   AS mes,
            categoria,
            COUNT(*)                           AS pedidos_mes,
            ROUND(SUM(valor_total), 2)         AS receita_mes
        FROM vw_pedidos_bronze
        WHERE status_pedido = 'concluido'
        GROUP BY 1, 2
    ),
    com_crescimento AS (
        SELECT
            mes,
            categoria,
            pedidos_mes,
            receita_mes,
            LAG(receita_mes) OVER (
                PARTITION BY categoria
                ORDER BY mes
            )                                  AS receita_mes_anterior,
            ROUND(
                100.0 * (receita_mes - LAG(receita_mes) OVER (
                    PARTITION BY categoria ORDER BY mes
                )) / NULLIF(LAG(receita_mes) OVER (
                    PARTITION BY categoria ORDER BY mes
                ), 0), 2
            )                                  AS crescimento_pct
        FROM vendas_mensais
    )
    SELECT *
    FROM com_crescimento
    WHERE mes >= '2022-01-01'
      AND mes < '2022-04-01'   -- Primeiros 3 meses para visualização
    ORDER BY categoria, mes
""").df()

In [12]:
print("Crescimento Mensal utilizando consulta da view de Receita por Categoria (Jan-Mar 2022):")
print(resultado_view.to_string(index=False))

Crescimento Mensal utilizando consulta da view de Receita por Categoria (Jan-Mar 2022):
       mes   categoria  pedidos_mes  receita_mes  receita_mes_anterior  crescimento_pct
2022-01-01   Alimentos        13367  73087878.22                   NaN              NaN
2022-02-01   Alimentos        12107  66861956.30           73087878.22            -8.52
2022-03-01   Alimentos        13355  74943624.38           66861956.30            12.09
2022-01-01 Eletrônicos        13478  73343279.53                   NaN              NaN
2022-02-01 Eletrônicos        12221  67857475.37           73343279.53            -7.48
2022-03-01 Eletrônicos        13554  74254134.56           67857475.37             9.43
2022-01-01    Esportes        13276  73153747.01                   NaN              NaN
2022-02-01    Esportes        12047  65934309.94           73153747.01            -9.87
2022-03-01    Esportes        13357  73474186.63           65934309.94            11.44
2022-01-01      Livros        13

In [13]:
# Célula 7 — Exportar resultado de consulta diretamente para Parquet no MinIO
# O DuckDB pode escrever resultados diretamente em S3 com COPY TO
con.execute("""
    COPY (
        SELECT
            categoria,
            regiao,
            DATE_TRUNC('month', data_pedido)  AS mes_referencia,
            COUNT(*)                          AS total_pedidos,
            ROUND(SUM(valor_total), 2)        AS receita_total,
            ROUND(AVG(valor_total), 2)        AS ticket_medio,
            COUNT(DISTINCT id_cliente)        AS clientes_unicos
        FROM vw_pedidos_bronze
        WHERE status_pedido = 'concluido'
        GROUP BY 1, 2, 3
        ORDER BY mes_referencia, categoria, regiao
    )
    TO 's3://bronze/ecommerce_agregado/resumo_mensal.parquet'
    (FORMAT PARQUET, COMPRESSION SNAPPY)
""")
# Verificar o arquivo criado
info = con.execute("""
    SELECT COUNT(*) AS linhas, MIN(mes_referencia) AS inicio, MAX(mes_referencia) AS fim
    FROM read_parquet('s3://bronze/ecommerce_agregado/resumo_mensal.parquet')
""").df()
print("✅ Arquivo exportado para o MinIO!")
print(info.to_string(index=False))

✅ Arquivo exportado para o MinIO!
 linhas     inicio        fim
    300 2022-01-01 2022-12-01
